# Parking Lot Occupancy Detection
## Hybrid EfficientNet-B0 + MobileViT Block
**ECE 4990 Final Project — Cal Poly Pomona**
**Authors: Francisco Pulido, Richard Pablo**

This notebook trains and evaluates:
- **Baseline:** EfficientNet-B0
- **Proposed:** EfficientNet-B0 + MobileViT hybrid

Datasets: PKLot, CNRPark-EXT, Parking Slot Classification Dataset

## 1. Install Dependencies

In [ ]:
!pip install -q timm einops kaggle matplotlib seaborn scikit-learn

## 2. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 3. Download Datasets from Kaggle

Upload your `kaggle.json` from: https://www.kaggle.com/settings -> API -> Create New Token

## 3. Download Datasets

**No Kaggle account or API key needed.** Datasets are downloaded directly from public URLs below.

In [ ]:
# ─── DATASET SETUP ──────────────────────────────────────────────────────────
# Download all 3 datasets from Kaggle (free account required to download):
#   PKLot:    https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset
#   CNRPark:  https://www.kaggle.com/datasets/ddsshubham/cnrpark-ext
#   ParkSlot: https://www.kaggle.com/datasets/basabbose/parking-slot-classification-dataset
#
# OPTION A — Upload zip files manually (easiest, no API key needed):
#   1. Download each dataset zip from the links above
#   2. Run the cell below to upload them directly to Colab
#
# OPTION B — Use Kaggle API (if you have kaggle.json):
#   Uncomment the API section at the bottom of this cell
# ─────────────────────────────────────────────────────────────────────────────

import os, zipfile
from pathlib import Path

os.makedirs('data/pklot',       exist_ok=True)
os.makedirs('data/cnrpark',     exist_ok=True)
os.makedirs('data/parking_slot',exist_ok=True)

# --- OPTION A: Upload zip files ---
from google.colab import files
print('Upload your dataset zip files (PKLot, CNRPark-EXT, ParkingSlot):')
uploaded = files.upload()  # Select all 3 zips at once

extract_map = {
    'pklot':        'data/pklot',
    'cnrpark':      'data/cnrpark',
    'cnr':          'data/cnrpark',
    'parking_slot': 'data/parking_slot',
    'parking-slot': 'data/parking_slot',
    'basabbose':    'data/parking_slot',
}

for fname in uploaded.keys():
    dest = next((v for k,v in extract_map.items() if k in fname.lower()), 'data/misc')
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(fname, 'r') as z: z.extractall(dest)
    print(f'Extracted {fname} -> {dest}')

print('Datasets ready!')

# --- OPTION B: Kaggle API (uncomment if you have kaggle.json) ---
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ammarnassanalhajali/pklot-dataset --unzip -p data/pklot
# !kaggle datasets download -d ddsshubham/cnrpark-ext --unzip -p data/cnrpark
# !kaggle datasets download -d basabbose/parking-slot-classification-dataset --unzip -p data/parking_slot


## 4. Dataset Loader

In [ ]:
def find_class_folders(root):
    root = Path(root)
    occ_names = {'occupied','busy','1','car'}
    emp_names = {'empty','free','0','no_car'}
    occ_dirs, emp_dirs = [], []
    for d in root.rglob('*'):
        if d.is_dir():
            n = d.name.lower()
            if n in occ_names: occ_dirs.append(d)
            elif n in emp_names: emp_dirs.append(d)
    return occ_dirs, emp_dirs

def build_dataframe(root, name, max_per_class=2000):
    occ_dirs, emp_dirs = find_class_folders(root)
    rows = []
    exts = {'.jpg','.jpeg','.png','.bmp'}
    for d in occ_dirs:
        for p in d.iterdir():
            if p.suffix.lower() in exts: rows.append({'path':str(p),'label':1,'dataset':name})
    for d in emp_dirs:
        for p in d.iterdir():
            if p.suffix.lower() in exts: rows.append({'path':str(p),'label':0,'dataset':name})
    df = pd.DataFrame(rows)
    occ = df[df.label==1].sample(min(max_per_class,(df.label==1).sum()), random_state=SEED)
    emp = df[df.label==0].sample(min(max_per_class,(df.label==0).sum()), random_state=SEED)
    df = pd.concat([occ,emp]).sample(frac=1,random_state=SEED).reset_index(drop=True)
    print(f'{name}: {len(df)} imgs | occ={len(occ)} emp={len(emp)}')
    return df

df_pklot   = build_dataframe('data/pklot',       'PKLot')
df_cnrpark = build_dataframe('data/cnrpark',      'CNRPark-EXT')
df_slot    = build_dataframe('data/parking_slot', 'ParkingSlot')
df_all = pd.concat([df_pklot,df_cnrpark,df_slot]).sample(frac=1,random_state=SEED).reset_index(drop=True)
print(f'Total: {len(df_all)}')

In [ ]:
def split_df(df):
    train,test = train_test_split(df,test_size=0.15,random_state=SEED,stratify=df['label'])
    train,val  = train_test_split(train,test_size=0.15/0.85,random_state=SEED,stratify=train['label'])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df,val_df,test_df = split_df(df_all)
print(f'Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
_,_,test_pklot   = split_df(df_pklot)
_,_,test_cnrpark = split_df(df_cnrpark)
_,_,test_slot    = split_df(df_slot)

In [ ]:
IMG_SIZE = 224
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class ParkingDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df=df; self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        try: img=Image.open(row['path']).convert('RGB')
        except: img=Image.fromarray(np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8))
        if self.transform: img=self.transform(img)
        return img, int(row['label'])

BATCH=32
train_loader   = DataLoader(ParkingDataset(train_df,   train_tfm),batch_size=BATCH,shuffle=True, num_workers=2,pin_memory=True)
val_loader     = DataLoader(ParkingDataset(val_df,     val_tfm),  batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
test_loader    = DataLoader(ParkingDataset(test_df,    val_tfm),  batch_size=BATCH,shuffle=False,num_workers=2)
loader_pklot   = DataLoader(ParkingDataset(test_pklot, val_tfm),  batch_size=BATCH,shuffle=False,num_workers=2)
loader_cnrpark = DataLoader(ParkingDataset(test_cnrpark,val_tfm), batch_size=BATCH,shuffle=False,num_workers=2)
loader_slot    = DataLoader(ParkingDataset(test_slot,  val_tfm),  batch_size=BATCH,shuffle=False,num_workers=2)
print('DataLoaders ready!')

## 5. Model Architectures
### 5A. Baseline: EfficientNet-B0

In [ ]:
class EfficientNetBaseline(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0',pretrained=pretrained,num_classes=0)
        self.classifier = nn.Sequential(nn.Dropout(0.3),nn.Linear(self.backbone.num_features,num_classes))
    def forward(self, x): return self.classifier(self.backbone(x))
print('Baseline defined.')

### 5B. Proposed: EfficientNet-B0 + MobileViT Block

After EfficientNet extracts local CNN features, a **MobileViT block** applies lightweight self-attention to capture global spatial context — helping detect occlusion and lighting variation that confuses pure CNN approaches.

In [ ]:
class MobileViTBlock(nn.Module):
    def __init__(self, dim, depth=2, num_heads=4, patch_size=2, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        patch_dim = dim*patch_size*patch_size
        self.local_conv = nn.Sequential(
            nn.Conv2d(dim,dim,3,padding=1,groups=dim,bias=False),
            nn.BatchNorm2d(dim),nn.SiLU(),nn.Conv2d(dim,dim,1,bias=False))
        self.norm = nn.LayerNorm(patch_dim)
        enc = nn.TransformerEncoderLayer(d_model=patch_dim,nhead=num_heads,
            dim_feedforward=int(patch_dim*mlp_ratio),dropout=dropout,
            activation='gelu',batch_first=True,norm_first=True)
        self.transformer = nn.TransformerEncoder(enc,num_layers=depth)
        self.proj = nn.Sequential(nn.Conv2d(dim,dim,1,bias=False),nn.BatchNorm2d(dim))

    def forward(self, x):
        B,C,H,W = x.shape; P=self.patch_size
        pad_h=(P-H%P)%P; pad_w=(P-W%P)%P
        if pad_h or pad_w: x=nn.functional.pad(x,(0,pad_w,0,pad_h))
        _,_,H2,W2 = x.shape
        local_out = self.local_conv(x)
        ph,pw = H2//P, W2//P
        patches = x.unfold(2,P,P).unfold(3,P,P)
        patches = patches.contiguous().view(B,C,ph,pw,P*P)
        patches = patches.permute(0,2,3,1,4).contiguous().view(B,ph*pw,C*P*P)
        patches = self.transformer(self.norm(patches))
        patches = patches.view(B,ph,pw,C,P,P).permute(0,3,1,4,2,5).contiguous().view(B,C,H2,W2)
        out = self.proj(local_out+patches)
        if pad_h or pad_w: out=out[:,:,:H,:W]
        return out + x[:,:,:H,:W]

class EfficientNetMobileViT(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0',pretrained=pretrained,num_classes=0,global_pool='')
        self.mobile_vit = MobileViTBlock(dim=320,depth=2,num_heads=4,patch_size=2)
        self.proj = nn.Sequential(nn.Conv2d(320,1280,1,bias=False),nn.BatchNorm2d(1280),nn.SiLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.3),nn.Linear(1280,num_classes))
    def forward(self, x):
        feat=self.backbone(x)
        if feat.dim()==2: return self.classifier(feat)
        feat=self.mobile_vit(feat); feat=self.proj(feat)
        return self.classifier(self.pool(feat).flatten(1))

print('Proposed model defined.')
with torch.no_grad():
    d=torch.randn(2,3,224,224)
    b,p=EfficientNetBaseline(),EfficientNetMobileViT()
    print(f'Baseline: {b(d).shape} | {sum(x.numel() for x in b.parameters())/1e6:.2f}M params')
    print(f'Proposed: {p(d).shape} | {sum(x.numel() for x in p.parameters())/1e6:.2f}M params')

## 6. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, model_name='model'):
    model=model.to(DEVICE)
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
    history={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[],'val_f1':[]}
    best_f1,best_state=0.0,None
    for epoch in range(epochs):
        model.train()
        t_loss,t_correct,t_total=0.0,0,0
        for imgs,labels in train_loader:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            optimizer.zero_grad()
            out=model(imgs); loss=criterion(out,labels)
            loss.backward(); optimizer.step()
            t_loss+=loss.item()*imgs.size(0)
            t_correct+=(out.argmax(1)==labels).sum().item()
            t_total+=imgs.size(0)
        model.eval()
        v_loss,all_preds,all_labels=0.0,[],[]
        with torch.no_grad():
            for imgs,labels in val_loader:
                imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
                out=model(imgs)
                v_loss+=criterion(out,labels).item()*imgs.size(0)
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        t_acc=t_correct/t_total
        v_acc=accuracy_score(all_labels,all_preds)
        v_f1=f1_score(all_labels,all_preds,average='macro')
        history['train_loss'].append(t_loss/t_total)
        history['val_loss'].append(v_loss/len(val_loader.dataset))
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['val_f1'].append(v_f1)
        if v_f1>best_f1:
            best_f1=v_f1
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        scheduler.step()
        print(f'[{model_name}] Ep{epoch+1:02d}/{epochs} | TrLoss:{t_loss/t_total:.4f} Acc:{t_acc:.4f} | ValLoss:{v_loss/len(val_loader.dataset):.4f} Acc:{v_acc:.4f} F1:{v_f1:.4f}')
    model.load_state_dict(best_state)
    torch.save(best_state,f'{model_name}_best.pth')
    print(f'Best Val F1: {best_f1:.4f}')
    return model, history

In [ ]:
print('='*60+'\nTRAINING BASELINE: EfficientNet-B0\n'+'='*60)
baseline_model, baseline_history = train_model(
    EfficientNetBaseline(),train_loader,val_loader,
    epochs=10,lr=1e-4,model_name='efficientnet_baseline')

In [ ]:
print('='*60+'\nTRAINING PROPOSED: EfficientNet-B0 + MobileViT\n'+'='*60)
proposed_model, proposed_history = train_model(
    EfficientNetMobileViT(),train_loader,val_loader,
    epochs=10,lr=1e-4,model_name='efficientnet_mobilevit')

## 7. Evaluation

In [ ]:
def evaluate(model, loader, name='Test'):
    model.eval()
    all_preds,all_labels=[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            preds=model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(labels.numpy())
    acc=accuracy_score(all_labels,all_preds)
    f1=f1_score(all_labels,all_preds,average='macro')
    print(f'{name:20s} | Acc:{acc:.4f} F1:{f1:.4f}')
    return acc,f1,all_labels,all_preds

print('\n=== BASELINE ===')
b_test_acc,b_test_f1,b_labels,b_preds=evaluate(baseline_model,test_loader,'Combined Test')
b_pklot_acc,b_pklot_f1,*_=evaluate(baseline_model,loader_pklot,'PKLot')
b_cnr_acc,b_cnr_f1,*_=evaluate(baseline_model,loader_cnrpark,'CNRPark-EXT')
b_slot_acc,b_slot_f1,*_=evaluate(baseline_model,loader_slot,'ParkingSlot')
print('\n=== PROPOSED ===')
p_test_acc,p_test_f1,p_labels,p_preds=evaluate(proposed_model,test_loader,'Combined Test')
p_pklot_acc,p_pklot_f1,*_=evaluate(proposed_model,loader_pklot,'PKLot')
p_cnr_acc,p_cnr_f1,*_=evaluate(proposed_model,loader_cnrpark,'CNRPark-EXT')
p_slot_acc,p_slot_f1,*_=evaluate(proposed_model,loader_slot,'ParkingSlot')

## 8. Results Table & Plots

In [ ]:
results=pd.DataFrame({'Dataset':['PKLot','CNRPark-EXT','ParkingSlot','Combined'],
    'Baseline Acc':[b_pklot_acc,b_cnr_acc,b_slot_acc,b_test_acc],
    'Baseline F1': [b_pklot_f1, b_cnr_f1, b_slot_f1, b_test_f1],
    'Proposed Acc':[p_pklot_acc,p_cnr_acc,p_slot_acc,p_test_acc],
    'Proposed F1': [p_pklot_f1, p_cnr_f1, p_slot_f1, p_test_f1]}).round(4)
print(results.to_string(index=False))
results.to_csv('results_table.csv',index=False)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,m,t in zip(axes,['val_loss','val_acc','val_f1'],['Val Loss','Val Accuracy','Val F1']):
    ax.plot(range(1,11),baseline_history[m],'b-o',label='Baseline',linewidth=2)
    ax.plot(range(1,11),proposed_history[m],'r-s',label='Proposed',linewidth=2)
    ax.set_title(t,fontweight='bold'); ax.legend(); ax.grid(True,alpha=0.3)
plt.suptitle('Training Curves',fontsize=14,fontweight='bold',y=1.02)
plt.tight_layout(); plt.savefig('training_curves.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
datasets=['PKLot','CNRPark-EXT','ParkingSlot','Combined']
x=np.arange(len(datasets)); w=0.35
bf=[b_pklot_f1,b_cnr_f1,b_slot_f1,b_test_f1]
pf=[p_pklot_f1,p_cnr_f1,p_slot_f1,p_test_f1]
b1=ax.bar(x-w/2,bf,w,label='Baseline',color='steelblue',alpha=0.85)
b2=ax.bar(x+w/2,pf,w,label='Proposed',color='tomato',alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(datasets)
ax.set_ylabel('Macro F1'); ax.set_ylim(0,1.05)
ax.set_title('F1-Score Comparison',fontweight='bold'); ax.legend(); ax.grid(axis='y',alpha=0.3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{bar.get_height():.3f}',ha='center',va='bottom',fontsize=9)
plt.tight_layout(); plt.savefig('f1_comparison.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,la,pr,ti in zip(axes,[b_labels,p_labels],[b_preds,p_preds],['Baseline','Proposed']):
    sns.heatmap(confusion_matrix(la,pr),annot=True,fmt='d',cmap='Blues',ax=ax,
        xticklabels=['Empty','Occupied'],yticklabels=['Empty','Occupied'])
    ax.set_title(ti,fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle('Confusion Matrices',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrices.png',dpi=150,bbox_inches='tight'); plt.show()

## 9. Hyperparameter Ablation Study

In [ ]:
print('Learning Rate Ablation...')
lr_results={}
for lr in [1e-3,5e-4,1e-4,5e-5]:
    m=EfficientNetMobileViT().to(DEVICE)
    opt=optim.Adam(m.parameters(),lr=lr); crit=nn.CrossEntropyLoss(); m.train()
    for _ in range(3):
        for imgs,labels in train_loader:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            opt.zero_grad(); loss=crit(m(imgs),labels); loss.backward(); opt.step()
    _,f1,_,_=evaluate(m,val_loader,f'lr={lr}'); lr_results[lr]=f1
fig,ax=plt.subplots(figsize=(7,4))
ax.plot([str(l) for l in lr_results],list(lr_results.values()),'go-',linewidth=2,markersize=8)
ax.set_xlabel('Learning Rate'); ax.set_ylabel('Val F1')
ax.set_title('Effect of Learning Rate',fontweight='bold'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('lr_ablation.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
print('Batch Size Ablation...')
bs_results={}
for bs in [16,32,64]:
    ltmp=DataLoader(ParkingDataset(train_df,train_tfm),batch_size=bs,shuffle=True,num_workers=2)
    m=EfficientNetMobileViT().to(DEVICE)
    opt=optim.Adam(m.parameters(),lr=1e-4); crit=nn.CrossEntropyLoss(); m.train()
    for _ in range(3):
        for imgs,labels in ltmp:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            opt.zero_grad(); loss=crit(m(imgs),labels); loss.backward(); opt.step()
    _,f1,_,_=evaluate(m,val_loader,f'batch={bs}'); bs_results[bs]=f1
fig,ax=plt.subplots(figsize=(6,4))
ax.bar([str(b) for b in bs_results],list(bs_results.values()),color=['#4C72B0','#DD8452','#55A868'],alpha=0.85)
ax.set_xlabel('Batch Size'); ax.set_ylabel('Val F1')
ax.set_title('Effect of Batch Size',fontweight='bold'); ax.set_ylim(0,1.0); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('bs_ablation.png',dpi=150,bbox_inches='tight'); plt.show()

## 10. Classification Reports

In [ ]:
print('=== BASELINE ===')
print(classification_report(b_labels,b_preds,target_names=['Empty','Occupied']))
print('=== PROPOSED ===')
print(classification_report(p_labels,p_preds,target_names=['Empty','Occupied']))

## 11. Download All Results

In [ ]:
from google.colab import files
import zipfile
output_files=['results_table.csv','training_curves.png','f1_comparison.png',
    'confusion_matrices.png','lr_ablation.png','bs_ablation.png',
    'efficientnet_baseline_best.pth','efficientnet_mobilevit_best.pth']
with zipfile.ZipFile('parking_results.zip','w') as zf:
    for f in output_files:
        if os.path.exists(f): zf.write(f)
files.download('parking_results.zip')
print('Done!')